# Lab 10b: reverse stress testing

**AI-powered Investment and Risk Management (25882), Lecture 10: AI in Risk Forecasting and Stress Testing**

An ordinary stress test asks: *in this scenario, how much would we lose?* A reverse stress test turns the question round: **what is the most plausible scenario that would make us lose this much?** The ECB's 2026 thematic stress test asked 110 banks exactly that, for a 300-basis-point fall in their capital ratio.

**You will**

- solve the reverse stress test in closed form for a linear portfolio, and watch the answer change with the history it learns from;
- add a small, profitable-looking short put and see the closed form fail;
- search instead, by optimisation and by sampling from the generators of Lab 10a;
- turn thousands of breaching months into three narratives a board could discuss.

**Data.** As in Lab 10a: eight US industry portfolios, long Treasuries and corporate bonds, monthly 1970–2025.

**Runtime.** Well under a minute on a laptop.

## Setup

In [ ]:
import os
import urllib.request

if not os.path.exists("airm10.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/VitaliAlexeev/AI_Investments_2026/main/airm10.py", "airm10.py")

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import multivariate_normal, multivariate_t, norm
from sklearn.cluster import KMeans

import airm10 as A

## 1. The portfolio and the loss

The portfolio holds 60% in the eight industries (equal weights) and 40% in long Treasuries. The loss to reach is 10% in a month: for this portfolio, a once-in-a-generation month.

In [ ]:
X = A.asset_panel()
w = A.benchmark_portfolios().loc["60/40"].values
L = -10.0
print(f"{len(X)} months, {X.index[0]} to {X.index[-1]}; worst month for this portfolio: {(X.values @ w).min():.1f}%")

## 2. The closed form, and why history matters

Treat the ten monthly returns as jointly normal. Plausibility is the Mahalanobis distance from the average month, and the most plausible breaching month has a closed form. We solve it with three histories.

<details>
<summary><b>The maths: the most plausible breach and its dual</b></summary>

With $x \sim N(\mu, \Sigma)$ and portfolio weights $w$, minimise $(x-\mu)^\top\Sigma^{-1}(x-\mu)$ subject to $w^\top x = L$. The Lagrangian gives

$$x^{*} = \mu + \frac{L - w^\top\mu}{w^\top\Sigma w}\,\Sigma w, \qquad d^{*} = \frac{w^\top\mu - L}{\sqrt{w^\top\Sigma w}} .$$

For a linear portfolio, the probability of a loss at least this large is $\Phi(-d^{*})$. Turned round (Breuer et al., 2009): among scenarios no further than $k$ from the centre, the worst loss is $w^\top\mu - k\sqrt{w^\top\Sigma w}$.
</details>

In [ ]:
def most_plausible(mu, S, w, L):
    return mu + (L - w @ mu) * (S @ w) / (w @ S @ w)

REGIMES = {"1970-1999": ("1970-01", "1999-12"), "2000-2021": ("2000-01", "2021-12"),
           "1970-2025": ("1970-01", "2025-12")}
closed, rows = {}, []
for name, (a, b) in REGIMES.items():
    E = X.loc[a:b]
    mu_r, S_r = E.mean().values, E.cov().values
    x = most_plausible(mu_r, S_r, w, L)
    d = A.mahalanobis_rows(x, mu_r, S_r)[0]
    closed[name] = x
    rows.append({"History": name, "Distance": d, "Probability of a month this bad": norm.cdf(-d),
                 "Worst loss within distance 3 (%)": A.worst_within(mu_r, S_r, w, 3.0)})
closed_table = pd.DataFrame(rows).set_index("History")
display(closed_table.round(4))
A.scenario_bars(closed, f"Most plausible month in which the portfolio loses {abs(L):.0f}%").show()

Learning from 1970–99, a 10% loss sits at distance 2.83, about one month in 440, and Treasuries fall with equities (−4.5%). Learning from 2000–21, the same loss sits at distance 4.05, about one month in 39,000, and Treasuries barely move (−1.2%).

Read the last column the other way round. Within a plausibility budget of distance 3, the worst month is −10.6% under the 1970–99 history but only −7.2% under 2000–21. The history you learn from decides how bad "plausible" can get.

## 3. A small, profitable-looking short put

Now suppose the portfolio also sells one-month puts on its equity basket, struck 8% below the current level, on half its value. Priced at 16% implied volatility, the put earns a premium every quiet month and loses fast below the strike. The portfolio is no longer linear in the returns, so the closed form no longer applies.

In [ ]:
STRIKE, NOTIONAL = -8.0, 0.5
premium = A.bs_put_premium(1 + STRIKE / 100, vol=0.16)

def pnl(Z):
    return A.overlay_pnl(Z, w, strike=STRIKE, notional=NOTIONAL, premium=premium)

E = X.loc["1970-01":"2025-12"]
mu, S = E.mean().values, E.cov().values
print(f"Premium: {premium:.3f}% of notional; income to the portfolio: {NOTIONAL * premium:.3f}% a month")
A.payoff_figure(mu, w, pnl, STRIKE).show()

## 4. Searching by optimisation

Minimise the distance subject to the loss constraint numerically, starting from the linear answer and from the average month.

In [ ]:
x_lin = most_plausible(mu, S, w, L)
x_opt = A.reverse_stress_opt(mu, S, pnl, L, starts=[x_lin, mu])
d_lin, d_opt = A.mahalanobis_rows(np.vstack([x_lin, x_opt]), mu, S)
print(f"Ignoring the put:  distance {d_lin:.2f}, but with the put that month loses {pnl(x_lin)[0]:.1f}%")
print(f"Searching with it: distance {d_opt:.2f}, loss {pnl(x_opt)[0]:.1f}%")
A.scenario_bars({"Ignoring the put": x_lin, "With the put (optimised)": x_opt},
                "Most plausible way to lose 10% in a month").show()

Zg = np.random.default_rng(1).multivariate_normal(mu, S, size=200_000)
p_without, p_with = np.mean(Zg @ w <= L), np.mean(pnl(Zg) <= L)
ratio = (f"{p_with / p_without:.1f} times as likely" if p_without > 0
         else "no breaches at all without the put in these draws")
print(f"Chance of losing {abs(L):.0f}% or more in a month (Gaussian, 1970-2025): "
      f"{p_without:.3%} without the put, {p_with:.3%} with it ({ratio})")

Ignoring the put, the most plausible 10% loss is at distance 3.19, and that same month would actually return −13.1% once the put is counted. Searching with the put, a 10% loss is reached at distance 2.63, with equities down about 12%, just past the strike.

Sampling from the same normal distribution, a month that bad happens 6.4 times as often with the put as without it (0.46% against 0.07%). The put pays 0.03% a month for that privilege.

## 5. Searching by sampling: the generators of Lab 10a

When the generator is not normal there is no formula, and often no clean optimisation. Sample-and-rank works for any generator: draw many months, keep those that breach, and rank them by their density under the generator.

<details>
<summary><b>The maths: why the best sample is not the optimum</b></summary>

In ten dimensions, typical draws from a normal distribution lie about $\sqrt{10}\approx 3.2$ units from the centre: most of the probability sits in a thin shell, not near the peak. So the most plausible *sampled* breach usually sits further out than the optimum of Section 4. Use optimisation for the single most plausible point and samples for the spread of possibilities, which is what narratives are made of.
</details>

In [ ]:
N = 200_000
mu_t, shape_t, df_t = A.fit_student_t(E)
mixture, _ = A.fit_mixture(E)
gens = {
    "Gaussian": (Zg, multivariate_normal(mu, S).logpdf),
    f"Student t (df {df_t})": (multivariate_t(mu_t, shape_t, df=df_t).rvs(size=N, random_state=np.random.default_rng(2)),
                               multivariate_t(mu_t, shape_t, df=df_t).logpdf),
    f"Mixture ({mixture.n_components} regimes)": (mixture.sample(N)[0], mixture.score_samples),
}
rows, best = [], {}
for name, (Z, logdens) in gens.items():
    hit = Z[pnl(Z) <= L]
    top = hit[np.argmax(logdens(hit))]
    best[name] = top
    rows.append({"Generator": name, "Breach probability": len(hit) / len(Z),
                 "Most likely breach: equities (%)": top[:8].mean(), "Treasuries (%)": top[8],
                 "Corporate bonds (%)": top[9], "Distance (normal yardstick)": A.mahalanobis_rows(top, mu, S)[0]})
search_table = pd.DataFrame(rows).set_index("Generator")
display(search_table.round(4))

mix_name = f"Mixture ({mixture.n_components} regimes)"
Zm = gens[mix_name][0]
A.breach_cloud(Zm, pnl(Zm), L, {"optimised (normal)": x_opt, "most likely (mixture)": best[mix_name]}).show()

The chance of a 10% month is 0.46% under the Gaussian, 0.90% under the Student *t* and 1.21% under the mixture: the fatter-tailed generators make it 1.9 and 2.6 times as likely. 99% of the mixture's breaches come from its volatile regime.

## 6. From scenarios to narratives

A board cannot read 2,413 scenarios. Clustering the mixture's breaching months into three groups, then reading each group's average, gives three stories. They are ordered here by what Treasuries do.

In [ ]:
hit = Zm[pnl(Zm) <= L]
regime = mixture.predict(hit)
print("Breaches by regime:", np.bincount(regime, minlength=mixture.n_components),
      "| regime weights:", mixture.weights_.round(3))

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit((hit - mu) / E.std().values)
order = np.argsort([hit[km.labels_ == c, 8].mean() for c in range(3)])
narratives = {f"Narrative {i + 1} ({(km.labels_ == c).sum()} months)": hit[km.labels_ == c].mean(axis=0)
              for i, c in enumerate(order)}
A.scenario_bars(narratives, "Three ways to lose 10%: the average breaching month in each cluster").show()
display(pd.DataFrame(narratives, index=A.LAB_ASSETS).T.round(1))

- **Narrative 1** (726 months): stocks and bonds fall together, the pattern of 2022 and of the 1970s. Equities −14.6%, Treasuries −6.0%, corporate bonds −7.4%.
- **Narrative 2** (955 months): a broad equity sell-off with Treasuries roughly flat. Equities −14.4%, Treasuries −1.4%, corporate bonds −2.4%.
- **Narrative 3** (732 months): a flight to quality: Treasuries rally, so equities must fall further to deliver the same loss. Equities −18.2%, Treasuries +3.3%, corporate bonds +0.3%.

This is what the ECB asked banks to produce in 2026, and where it found the weakest work: turning narratives into consistent numbers. Here the numbers come first and the narratives are read off them. The bridge to Lecture 9 is direct: a counterfactual explanation asks for the smallest plausible change that flips one decision, and a reverse stress test asks the same question of a portfolio.

## What to notice

1. **The closed form is exact but narrow.** It needs a linear portfolio and a normal world, and it inherits the history you estimate it on.
2. **Small non-linearities move the answer.** A put earning 0.03% a month brings a 10% loss from distance 3.19 to 2.63, making it 6.4 times as likely.
3. **Search is where AI earns its place:** optimisation for the single most plausible month, generators for the spread of possibilities, clustering to turn that spread into narratives.

## Exercises

**Core 1.** Set `L = -15.0` and rerun. Does the closed-form scenario change shape, or only scale? What happens to the optimised scenario with the put?

**Core 2.** Use only 2000–2021 for sections 3 to 6 by changing `E`. Which narrative weakens or disappears, and why?

**Stretch 1.** Write one sentence for each narrative, as a bank would for the ECB. Then check internal consistency: are the signs in each average month consistent with the correlations in the regime most breaches come from?

**Stretch 2.** Importance sampling. Draw only from the mixture's volatile regime, reweight each draw by the ratio of the full mixture's density to that regime's density, and compare the precision of the breach-probability estimate with plain sampling.